# Lesson 4: a CNOT on a 2D cluster patch

Two wires that never touch can't make entanglement between logical qubits. In a 2D cluster the wires are joined by
vertical edges, and **an edge between the qubits currently holding two logical qubits is a CZ gate between them**, for free,
because all the CZs were applied when the resource was made.

Then $\mathrm{CNOT} = (I\otimes H)\,CZ\,(I\otimes H)$, and $H = J(0)$ is one wire step. So we need:

```
   control  c                     c stays put (input = output)
            |
   t0 ---- t1 ---- t2             target: t0 -> t1 (H), CZ with c, t1 -> t2 (H)
```

This 4-qubit pattern is the standard CNOT of the measurement calculus
([Danos, Kashefi & Panangaden, arXiv:0704.1263](https://arxiv.org/abs/0704.1263)). We'll cut it out of a $2\times3$
patch of the 2D cluster using the Z-measurement trick from lesson 1:

```
   (0,0) -- (0,1) -- (0,2)        (0,0), (0,2): measure Z (deletes them)
     |        |        |          (0,1): control
   (1,0) -- (1,1) -- (1,2)        (1,0) -> (1,1) -> (1,2): target wire
```
The original paper's CNOT uses a bigger 15-qubit patch with all qubits in place
([arXiv:quant-ph/0301052](https://arxiv.org/abs/quant-ph/0301052)); the small one is easier to read.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from mbqc import *

rng = np.random.default_rng(2026)

## Step by step

The byproduct rule for the CZ edge: $CZ\,(X\otimes I) = (X\otimes Z)\,CZ$. An $X$ byproduct on the target when the
edge acts becomes a $Z$ byproduct on the control. `PauliFrame.cz` does exactly that.

In [2]:
psi = random_state(2, rng)                  # order: (control, target)
c, (t0, t1, t2) = (0, 1), [(1, 0), (1, 1), (1, 2)]
edges = grid_edges(2, 3)

state = QubitState(psi, [c, t0])
for q in [(0, 0), (0, 2), t1, t2]:
    state.add(q)
for e in edges:
    state.cz(*e)                            # the 2D cluster patch, made once, up front

frame = PauliFrame()
# carve: Z measurements remove the corners and leave Z byproducts on their neighbours
s = state.measure_z((0, 0), rng=rng); frame.z_measured(s, [c, t0])
s = state.measure_z((0, 2), rng=rng); frame.z_measured(s, [c, t2])
# target: H from t0 to t1, the c-t1 edge acts as CZ, H from t1 to t2
s = state.measure_xy(t0, frame.angle(t0, 0.0), rng=rng); frame.teleport(t0, t1, s)
frame.cz(c, t1)
s = state.measure_xy(t1, frame.angle(t1, 0.0), rng=rng); frame.teleport(t1, t2, s)
frame.undo(state, c), frame.undo(state, t2)

CNOT = np.array([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0]])
print("fidelity with CNOT|psi>:", round(fidelity(state.vector([c, t2]), CNOT @ psi), 12))

fidelity with CNOT|psi>: 1.0


## All 16 branches, and a Bell state

`run_cnot_patch` packages the cell above. Every outcome branch gives the right answer, and feeding in $|+\rangle|0\rangle$ makes a Bell pair:
the patch really creates entanglement between two logical qubits.

In [3]:
import itertools

measured = [(0, 0), (0, 2), (1, 0), (1, 1)]
fids = [fidelity(run_cnot_patch(psi, outcomes=dict(zip(measured, o)))[0], CNOT @ psi)
        for o in itertools.product((0, 1), repeat=4)]
print("min fidelity over 16 branches:", round(min(fids), 12))

out, _ = run_cnot_patch(np.kron(PLUS, ZERO), rng=rng)
print("|+>|0> -> ", np.round(out, 4), "  entanglement:", QubitState(out, ["c", "t"]).entropy(["c"]), "bit")

min fidelity over 16 branches: 1.0
|+>|0> ->  [0.7071+0.j 0.    +0.j 0.    +0.j 0.7071+0.j]   entanglement: 1.0 bit


**Takeaways**
* In a 2D cluster, vertical edges are CZ gates between logical qubits, already "applied".
* Z measurements carve the patch you need; XY measurements do the gates.
* Byproducts propagate through CZ edges ($X$ on one side gives $Z$ on the other) and are tracked, not fixed.